# Кластеризация клиентов торгового центра
Делим клиентов на группы по доходу и тратам тремя методами: K-means, иерархическая кластеризация, DBSCAN.

## 1. Данные

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score

df = pd.read_csv('Mall_Customers.csv')
df.head()

In [ ]:
df.describe()

Берём два признака: доход и траты. Масштабируем их, чтобы оба влияли одинаково.

In [ ]:
X = df[['Annual Income (k$)', 'Spending Score (1-100)']]
X_scaled = StandardScaler().fit_transform(X)

plt.scatter(X['Annual Income (k$)'], X['Spending Score (1-100)'])
plt.xlabel('Доход (k$)')
plt.ylabel('Траты (1-100)')
plt.show()

## 2. K-means
Сначала выберем число кластеров методом локтя: ищем место, где кривая перестаёт резко падать.

In [ ]:
wcss = []
for k in range(1, 11):
    wcss.append(KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_scaled).inertia_)

plt.plot(range(1, 11), wcss, marker='o')
plt.xlabel('Число кластеров')
plt.ylabel('WCSS')
plt.show()

Излом на **k = 5**.

In [ ]:
df['kmeans'] = KMeans(n_clusters=5, n_init=10, random_state=42).fit_predict(X_scaled)

plt.scatter(X['Annual Income (k$)'], X['Spending Score (1-100)'], c=df['kmeans'], cmap='tab10')
plt.xlabel('Доход (k$)')
plt.ylabel('Траты (1-100)')
plt.title('K-means')
plt.show()

print('Silhouette:', round(silhouette_score(X_scaled, df['kmeans']), 3))

Средние значения по каждой группе:

In [ ]:
df.groupby('kmeans')[['Annual Income (k$)', 'Spending Score (1-100)']].mean().round(1)

## 3. Иерархическая кластеризация

In [ ]:
df['hier'] = AgglomerativeClustering(n_clusters=5).fit_predict(X_scaled)

plt.scatter(X['Annual Income (k$)'], X['Spending Score (1-100)'], c=df['hier'], cmap='tab10')
plt.xlabel('Доход (k$)')
plt.ylabel('Траты (1-100)')
plt.title('Иерархическая кластеризация')
plt.show()

print('Silhouette:', round(silhouette_score(X_scaled, df['hier']), 3))

## 4. DBSCAN
Число кластеров задавать не нужно. Выбросы получают метку -1.

In [ ]:
df['dbscan'] = DBSCAN(eps=0.3, min_samples=5).fit_predict(X_scaled)

plt.scatter(X['Annual Income (k$)'], X['Spending Score (1-100)'], c=df['dbscan'], cmap='tab10')
plt.xlabel('Доход (k$)')
plt.ylabel('Траты (1-100)')
plt.title('DBSCAN')
plt.show()

noise = df['dbscan'] == -1
print('Кластеров:', df.loc[~noise, 'dbscan'].nunique())
print('Выбросов:', noise.sum())
print('Silhouette (без выбросов):', round(silhouette_score(X_scaled[~noise], df.loc[~noise, 'dbscan']), 3))

## 5. Выводы

- Оптимально **5 кластеров**: так показывает локоть.
- K-means и иерархическая кластеризация дали почти одинаковый результат (silhouette около 0.55).
- DBSCAN нашёл 7 групп и около 17% выбросов, его silhouette ниже; он зависит от параметров.
- Лучший вариант здесь — K-means: просто и каждому клиенту достаётся группа.

Группы клиентов:
1. средний доход, средние траты (самая большая группа);
2. высокий доход, высокие траты (самые ценные клиенты);
3. низкий доход, высокие траты;
4. высокий доход, низкие траты (можно привлечь предложениями);
5. низкий доход, низкие траты.